# Task 5 — Evaluate the Platform

Measures the production readiness of the platform on the Task 1 update release:

| Measurement | How |
|---|---|
| incremental update time | ingesting each update file into the pre-update Silver tables, vs re-ingesting the whole dataset |
| analytical refresh time | incremental Gold append + incremental product merge (Task 2) vs full Gold rebuild + full product build (Week 1/2) |
| storage overhead | size of every Delta table split into active data, retained history and transaction log; Week 3 tables; bytes written by incremental vs full refresh |
| validation overhead | same update ingested with **no validation**, the **Week 1** checks and the **Task 4** extended validation |
| monitoring overhead | the same pipelines with run metadata / validation reports / storage measurement switched **on and off**, plus micro-benchmarks of each monitoring write |

**Isolation.** Nothing here writes to the platform tables. The pre-update state is reconstructed once from the
platform (every Silver/Gold row whose `source_file` is not an update file) into `data/delta/_evaluation/baseline`.
Every measured run starts from a fresh clone of that baseline in `data/delta/_evaluation/run` (data files are
hard-linked, the transaction log is copied), so each run appends to a realistic, full-size table.

**Method.** One un-timed warm-up run per dataset (JVM, OS page cache), then `REPEATS` timed runs with the
variants interleaved, so drift affects every variant equally. Tables report the median; min/max are kept in the CSVs.
The code under test is the real implementation: the Week 1 ingestor (`src/ingestion/ingestor.py`), the Task 4
validation framework (loaded from `expanded_ingestion.ipynb`), the integration module
(`src/integration/integrated_taxi_trips.py`) and the Task 2 incremental product refresh
(loaded from `maintain_analytical_consistency.ipynb`).

Prerequisites: Week 1 ingestion, `generate_incremental_updates.ipynb` (update files ingested into the platform).

## 0. Setup

In [1]:
import os
import sys
import json
import time
import shutil
import datetime
from dataclasses import asdict, replace


def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


def load_notebook_cells(path, namespace, tag=None, contains=None):
    """Execute selected code cells of another notebook: cells tagged `tag`, or containing `contains`."""
    nb = json.load(open(path))
    loaded = 0
    for cell in nb["cells"]:
        source = "".join(cell["source"])
        if cell["cell_type"] != "code":
            continue
        if (tag and tag in cell.get("metadata", {}).get("tags", [])) or (contains and contains in source):
            exec(compile(source, f"{os.path.basename(path)}", "exec"), namespace)
            loaded += 1
    if not loaded:
        raise ValueError(f"no cells selected in {path}")
    return loaded


# Task 4 validation framework (setup, rule registry, schema checks, validator, validated ingestion pipeline).
# It also creates the Spark session and defines LayerPaths, PLATFORM, read_delta_spark, CONFIGS, ...
n = load_notebook_cells(os.path.join(PROJECT_ROOT, "notebooks", "week3", "expanded_ingestion.ipynb"),
                        globals(), tag="framework")
print(f"Loaded {n} framework cells from expanded_ingestion.ipynb")

# Task 2 incremental product refresh, executed in its own namespace so its globals can be redirected
T2 = {"spark": spark, "time": time, "PROJECT_ROOT": PROJECT_ROOT}
load_notebook_cells(os.path.join(PROJECT_ROOT, "notebooks", "week3", "maintain_analytical_consistency.ipynb"),
                    T2, contains="def refresh_products_incremental")

import src.ingestion.ingestor as week1_ingestor
import src.integration.integrated_taxi_trips as integration
from src.common.config_loader import load_dataset_configs

WEEK1_CONFIGS = load_dataset_configs()          # config/datasets.yaml (Week 1 rules)

EVAL_ROOT = os.path.join(PROJECT_ROOT, "data", "delta", "_evaluation")
BASELINE = LayerPaths(os.path.join(EVAL_ROOT, "baseline"))
RUN = LayerPaths(os.path.join(EVAL_ROOT, "run"))
RESULTS_DIR = os.path.join(PROJECT_ROOT, "reports", "week3_evaluation")
os.makedirs(RESULTS_DIR, exist_ok=True)

UPDATE_FILES = {
    "tripdata": os.path.join(PROJECT_ROOT, "data/raw/tripdata/tripdata_update_2024_04.parquet"),
    "weather": os.path.join(PROJECT_ROOT, "data/raw/weather/weather_update.csv"),
    "air_quality": os.path.join(PROJECT_ROOT, "data/raw/air_quality/air_quality_update.csv"),
}
UPDATE_MARKER = "_update"                        # update files are named *_update*
EVOLUTION_COLUMNS = ["humidity", "aqi"]          # added by the update release
DATASETS = list(UPDATE_FILES)
REPEATS = 3

missing = [p for p in UPDATE_FILES.values() if not os.path.exists(p)]
assert not missing, f"run generate_incremental_updates.ipynb first, missing: {missing}"

GOLD = "integrated_taxi_trips"
PRODUCTS = "data_products"
MB = 1024 * 1024


def gold_path(paths): return os.path.join(paths.root, "gold", GOLD)
def products_path(paths): return os.path.join(paths.root, "gold", PRODUCTS)


def timed(fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def median_table(df, index, columns, value="seconds"):
    return df.pivot_table(index=index, columns=columns, values=value, aggfunc="median")

26/09/27 00:29:43 WARN Utils: Your hostname, DESKTOP-821NIV4 resolves to a loopback address: 127.0.1.1; using 10.10.10.40 instead (on interface wlo1)
26/09/27 00:29:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 00:29:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /home/bogdan/repos/ID2221/LAB
Datasets: ['tripdata', 'weather', 'air_quality', 'taxi_zone_lookup']
Registered rule types: {'not_null': 'incomplete', 'cast_errors': 'invalid_value', 'valid_range': 'invalid_value', 'allowed_values': 'invalid_value', 'regex': 'invalid_value', 'expression': 'invalid_value', 'unique': 'duplicate', 'not_in_table': 'duplicate', 'reference': 'missing_reference'}
tripdata           16 rules: type_conformance, incomplete, duplicate_in_batch, duplicate_existing, invalid_trip_distance, invalid_passenger_count, invalid_pickup_year, invalid_pickup_month, invalid_vendor_id, invalid_payment_type, invalid_ratecode_id, dropoff_before_pickup, trip_longer_than_24h, missing_pickup_zone, missing_dropoff_zone, negative_fare
weather            12 rules: type_conformance, incomplete, duplicate_in_batch, duplicate_existing, invalid_temperature, invalid_relative_humidity, invalid_humidity, invalid_precipitation, invalid_wind_speed, invalid_wind_direction, invalid_p

## 1. Pre-update baseline

The platform already contains the update release (Task 1 ingested it). The state *before* the update is
reconstructed by dropping every row that came from an update file and the columns the update introduced.
Built once and reused on later runs of this notebook (set `REBUILD_BASELINE = True` to force).

In [2]:
REBUILD_BASELINE = False


def week2_products(gold_df):
    """The four Week 2 data products (same definitions as run_reusable_products.ipynb)."""
    gold_df = gold_df.filter(F.col("is_valid"))
    return {
        "daily_mobility_summary": gold_df.groupBy("event_date", "pickup_borough").agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
            F.round(F.sum("fare_amount"), 2).alias("total_revenue")),
        "taxi_zone_statistics": gold_df.groupBy("pickup_zone", "pickup_borough").agg(
            F.count("*").alias("total_pickups"),
            F.round(F.avg("fare_amount"), 2).alias("avg_pickup_fare"),
            F.round(F.avg("trip_distance"), 2).alias("avg_pickup_distance")),
        "weather_impact_summary": gold_df.groupBy("weather_condition_code").agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare")),
        "air_quality_impact_summary": T2["_pm25_bucket_col"](gold_df).groupBy("pm25_level").agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("air_quality_pm25"), 2).alias("avg_pm25"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance")),
    }


def build_products_full(gold_df, out_dir):
    for name, df in week2_products(gold_df).items():
        write_deltalake(os.path.join(out_dir, name), T2["add_audit_metadata"](df).toPandas(), mode="overwrite")


def build_baseline():
    shutil.rmtree(BASELINE.root, ignore_errors=True)
    counts = []
    for ds, cfg in CONFIGS.items():
        platform = read_delta_spark(PLATFORM.table("silver", ds))
        pre = platform.filter(~F.col("source_file").contains(UPDATE_MARKER)).drop(
            *[c for c in EVOLUTION_COLUMNS if c in platform.columns])
        written = write_delta(pre, BASELINE.table("silver", ds), mode="overwrite", partition_by=cfg.partition_by)
        counts.append({"table": f"silver.{ds}", "platform_rows": platform.count(), "baseline_rows": written})
    platform_gold = read_delta_spark(gold_path(PLATFORM))
    pre_gold = platform_gold.filter(~F.col("source_file").contains(UPDATE_MARKER))
    written = write_delta(pre_gold, gold_path(BASELINE), mode="overwrite", partition_by=["pickup_year", "pickup_month"])
    counts.append({"table": f"gold.{GOLD}", "platform_rows": platform_gold.count(), "baseline_rows": written})
    build_products_full(read_delta_spark(gold_path(BASELINE)), products_path(BASELINE))
    return pd.DataFrame(counts)


if REBUILD_BASELINE or not os.path.exists(os.path.join(gold_path(BASELINE), "_delta_log")):
    t, baseline_counts = timed(build_baseline)
    print(f"Baseline built in {t:.1f}s")
    baseline_counts["removed_by_update"] = baseline_counts["platform_rows"] - baseline_counts["baseline_rows"]
    display(baseline_counts)
else:
    print("Reusing existing baseline:", BASELINE.root)

Reusing existing baseline: /home/bogdan/repos/ID2221/LAB/data/delta/_evaluation/baseline


In [3]:
def _link_or_copy(src, dst):
    """Delta data files are immutable, so they can be hard-linked; the log is copied."""
    if f"{os.sep}_delta_log{os.sep}" in src:
        shutil.copy2(src, dst)
    else:
        os.link(src, dst)


def prepare_run(datasets=None, with_gold=False):
    """Fresh copy of the pre-update baseline in RUN (cost is not part of any measurement)."""
    shutil.rmtree(RUN.root, ignore_errors=True)
    for ds in set(CONFIGS if datasets is None else datasets) | {"taxi_zone_lookup"}:
        shutil.copytree(BASELINE.table("silver", ds), RUN.table("silver", ds), copy_function=_link_or_copy)
    if with_gold:
        shutil.copytree(gold_path(BASELINE), gold_path(RUN), copy_function=_link_or_copy)
        shutil.copytree(products_path(BASELINE), products_path(RUN), copy_function=_link_or_copy)


t, _ = timed(prepare_run, with_gold=True)
print(f"Cloning the full baseline (Silver + Gold + products) takes {t:.5f}s")

Cloning the full baseline (Silver + Gold + products) takes 0.00262s


## 2. Incremental update time, validation overhead and monitoring overhead

Each update file is ingested into a fresh clone of the pre-update Silver table with five variants:

| Variant | Validation | Monitoring |
|---|---|---|
| `no_validation` | none — Week 1 load/standardise/cast/transform, straight to Bronze + Silver | none |
| `week1` | Week 1 schema check, duplicate removal against Silver, `apply_quality_checks` (real `src` ingestor) | off |
| `week1+monitoring` | same | on (run metadata, storage measurement) |
| `task4` | Task 4 schema checks + all configured rules | off |
| `task4+monitoring` | same | on (run metadata, validation reports, storage measurement) |

*validation overhead* = variant − `no_validation`; *monitoring overhead* = `+monitoring` − same variant without.

In [4]:
def ingest_no_validation(ds, source):
    cfg = CONFIGS[ds]
    raw = load_source_file(spark, cfg, source)
    write_delta(add_lineage_columns(raw, source, cfg.schema_version), RUN.table("bronze", ds),
                mode="append", schema_mode="merge")
    df = standardize_columns(raw, cfg.column_mapping)
    df = normalize_missing_values(df)
    df = change_datatype(df, cfg.dtype_overrides)
    df = get_transform(cfg.transform_module)(df, spark)
    df = add_lineage_columns(df, source, cfg.schema_version).drop(CAST_ERRORS)
    inserted = write_delta(df, RUN.table("silver", ds), mode=cfg.write_mode,
                           partition_by=cfg.partition_by, schema_mode="merge")
    return {"inserted": inserted, "quarantined": 0, "ignored": 0}


def _week1_config(ds):
    """Week 1 DatasetConfig whose layer paths point to RUN instead of the platform."""
    cfg = replace(WEEK1_CONFIGS[ds])
    cfg.bronze_path = lambda: RUN.table("bronze", ds)
    cfg.silver_path = lambda: RUN.table("silver", ds)
    cfg.quarantine_path = lambda: RUN.table("quarantine", ds)
    return cfg


def _save_week1_metadata(metadata):
    write_deltalake(RUN.ingestion_runs, pd.DataFrame([asdict(metadata)]), mode="append", schema_mode="merge")


def ingest_week1(ds, source, record):
    # the Week 1 ingestor records its monitoring through these two module-level functions
    week1_ingestor.save_metadata = _save_week1_metadata if record else (lambda m: None)
    week1_ingestor.get_dir_size_bytes = get_dir_size_bytes if record else (lambda p: 0)
    m = week1_ingestor._ingest_one_file(spark, _week1_config(ds), source)
    assert m.status == "SUCCESS", m.error_message
    return {"inserted": m.records_valid_silver, "quarantined": m.records_quarantined,
            "ignored": m.records_duplicate_ignored}


def ingest_task4(ds, source, record):
    m = run_validated_ingestion(ds, source, paths=RUN, reference=RUN, record=record)
    assert m["status"] == "SUCCESS", m["error_message"]
    return {"inserted": m["records_valid_silver"], "quarantined": m["records_quarantined"],
            "ignored": m["records_duplicate_ignored"]}


VARIANTS = {
    "no_validation": lambda ds, src: ingest_no_validation(ds, src),
    "week1": lambda ds, src: ingest_week1(ds, src, record=False),
    "week1+monitoring": lambda ds, src: ingest_week1(ds, src, record=True),
    "task4": lambda ds, src: ingest_task4(ds, src, record=False),
    "task4+monitoring": lambda ds, src: ingest_task4(ds, src, record=True),
}

ingest_rows = []
for ds in DATASETS:
    source = UPDATE_FILES[ds]
    prepare_run([ds])
    VARIANTS["task4+monitoring"](ds, source)                       # warm-up, not measured
    for rep in range(REPEATS):
        for variant, fn in VARIANTS.items():
            prepare_run([ds])
            seconds, out = timed(fn, ds, source)
            ingest_rows.append({"dataset": ds, "variant": variant, "repeat": rep, "seconds": seconds, **out})
    print(f"{ds}: done")

ingest_times = pd.DataFrame(ingest_rows)
ingest_times.to_csv(os.path.join(RESULTS_DIR, "ingestion_variants.csv"), index=False)

print("Median seconds per update file")
ingest_median = median_table(ingest_times, "dataset", "variant")[list(VARIANTS)].round(2)
display(ingest_median)

print("Records written by each variant (first repeat)")
display(ingest_times[ingest_times["repeat"] == 0].set_index(["dataset", "variant"])[["inserted", "quarantined", "ignored"]])

tripdata: done
weather: done


air_quality: done
Median seconds per update file


variant,no_validation,week1,week1+monitoring,task4,task4+monitoring
dataset,,,,,
air_quality,0.87,113.35,109.85,4.50,3.45
tripdata,7.12,66.73,65.75,12.17,11.78
weather,0.74,2.05,2.07,1.89,1.81


Records written by each variant (first repeat)


inserted  quarantined  ignored
dataset     variant                                         
tripdata    no_validation       343583            0        0
            week1               293672            0    49911
            week1+monitoring    293672            0    49911
            task4               293655           17    49911
            task4+monitoring    293655           17    49911
weather     no_validation          168            0        0
            week1                  168            0        0
            week1+monitoring       168            0        0
            task4                  168            0        0
            task4+monitoring       168            0        0
air_quality no_validation          500            0        0
            week1                  474            0       26
            week1+monitoring       474            0       26
            task4                  474            0       26
            task4+monitoring       474            0       26

In [5]:
m = ingest_median
overhead = pd.DataFrame({
    "no_validation_s": m["no_validation"],
    "week1_validation_s": (m["week1"] - m["no_validation"]),
    "week1_validation_%": 100 * (m["week1"] - m["no_validation"]) / m["no_validation"],
    "task4_validation_s": (m["task4"] - m["no_validation"]),
    "task4_validation_%": 100 * (m["task4"] - m["no_validation"]) / m["no_validation"],
    "week1_monitoring_s": (m["week1+monitoring"] - m["week1"]),
    "task4_monitoring_s": (m["task4+monitoring"] - m["task4"]),
    "task4_monitoring_%": 100 * (m["task4+monitoring"] - m["task4"]) / m["task4"],
}).round(2)
overhead.to_csv(os.path.join(RESULTS_DIR, "validation_monitoring_overhead.csv"))
display(overhead)

,no_validation_s,week1_validation_s,week1_validation_%,task4_validation_s,task4_validation_%,week1_monitoring_s,task4_monitoring_s,task4_monitoring_%
dataset,,,,,,,,
air_quality,0.87,112.48,12928.74,3.63,417.24,-3.50,-1.05,-23.33
tripdata,7.12,59.61,837.22,5.05,70.93,-0.98,-0.39,-3.20
weather,0.74,1.31,177.03,1.15,155.41,0.02,-0.08,-4.23


**Note on the percentages.** `no_validation` does no duplicate check at all and takes well under a second for the two
small update files, so any fixed cost becomes hundreds or thousands of percent. The absolute seconds are the
meaningful figure; only tripdata is large enough for a percentage to be informative.

### Where the validation time goes

Task 4 validation of the tripdata update, one rule at a time (dry run, nothing written): the file is validated with
all rules, then again with each rule removed. The difference is that rule's cost. Rules that read other tables
(`not_in_table`, `reference`) are expected to dominate.

In [6]:
def validation_seconds(ds, rules):
    cfg, spec = CONFIGS[ds], dict(SPECS[ds])
    spec["validation"] = {"rules": rules}
    df = load_source_file(spark, cfg, UPDATE_FILES[ds])
    df = standardize_columns(df, cfg.column_mapping)
    df = normalize_missing_values(df)
    df = change_datatype(df, cfg.dtype_overrides)
    df = get_transform(cfg.transform_module)(df, spark)
    validator = ExtendedValidator(cfg, spec, RUN.root)
    t0 = time.perf_counter()
    validator.validate(df).groupBy("_v_status").count().collect()
    return time.perf_counter() - t0


prepare_run(["tripdata"])
all_rules = SPECS["tripdata"]["validation"]["rules"]
validation_seconds("tripdata", all_rules)                           # warm-up
base = min(validation_seconds("tripdata", all_rules) for _ in range(REPEATS))
rule_cost = []
for rule in all_rules:
    without = [r for r in all_rules if r is not rule]
    t = min(validation_seconds("tripdata", without) for _ in range(REPEATS))
    rule_cost.append({"rule": rule["name"], "type": rule["type"], "cost_s": round(base - t, 3)})
rule_cost = pd.DataFrame(rule_cost).sort_values("cost_s", ascending=False, ignore_index=True)
print(f"All {len(all_rules)} rules: {base:.2f}s (best of {REPEATS})")
rule_cost.to_csv(os.path.join(RESULTS_DIR, "rule_cost_tripdata.csv"), index=False)
display(rule_cost)

[Stage 2131:>                                                       (0 + 8) / 8]

All 16 rules: 6.26s (best of 3)


,rule,type,cost_s
0,duplicate_existing,not_in_table,1.257
1,type_conformance,cast_errors,0.369
2,incomplete,not_null,0.241
3,duplicate_in_batch,unique,0.237
4,negative_fare,valid_range,0.225
5,missing_dropoff_zone,reference,0.211
6,dropoff_before_pickup,expression,0.207
7,invalid_payment_type,allowed_values,0.179
8,invalid_passenger_count,valid_range,0.127
9,invalid_vendor_id,allowed_values,0.114


### Monitoring micro-benchmarks

The end-to-end difference above is small compared with run-to-run noise, so each monitoring action is also timed
on its own (`MICRO_REPEATS` times, median in milliseconds).

In [7]:
MICRO_REPEATS = 10
micro_dir = os.path.join(EVAL_ROOT, "micro")
shutil.rmtree(micro_dir, ignore_errors=True)

sample_meta = asdict(IngestionRunMetadata(run_id="x", dataset_name="tripdata", source_file="f",
                                          format="parquet", schema_version="1.0", status="SUCCESS"))
sample_reports = pd.DataFrame([{"run_id": "x", "dataset_name": "tripdata", "source_file": "f",
                                "rule_name": f"rule_{i}", "reason": f"rule_{i}", "category": "invalid_value",
                                "action": "quarantine", "violations": i, "reported_at": "now"} for i in range(15)])

actions = {
    "write run metadata row (ingestion_runs)":
        lambda: write_deltalake(os.path.join(micro_dir, "runs"), pd.DataFrame([sample_meta]), mode="append", schema_mode="merge"),
    "write validation report rows (validation_reports)":
        lambda: write_deltalake(os.path.join(micro_dir, "reports"), sample_reports, mode="append", schema_mode="merge"),
    "storage scan of silver.tripdata (get_dir_size_bytes)":
        lambda: get_dir_size_bytes(PLATFORM.table("silver", "tripdata")),
    "storage scan of silver.weather (get_dir_size_bytes)":
        lambda: get_dir_size_bytes(PLATFORM.table("silver", "weather")),
}
micro = []
for name, fn in actions.items():
    samples = [timed(fn)[0] * 1000 for _ in range(MICRO_REPEATS)]
    micro.append({"monitoring action": name, "median_ms": round(pd.Series(samples).median(), 2),
                  "max_ms": round(max(samples), 2)})
micro = pd.DataFrame(micro)

runs = read_delta_spark(os.path.join(micro_dir, "runs"))
t_query, _ = timed(lambda: runs.groupBy("dataset_name").agg(F.max("execution_time_seconds")).collect())
print(f"ingestion_runs after {MICRO_REPEATS} runs: {len(DeltaTable(os.path.join(micro_dir, 'runs')).file_uris())} files, "
      f"{get_dir_size_bytes(os.path.join(micro_dir, 'runs')) / 1024:.0f} KiB; a monitoring query takes {t_query*1000:.0f} ms")
micro.to_csv(os.path.join(RESULTS_DIR, "monitoring_micro.csv"), index=False)
display(micro)

ingestion_runs after 10 runs: 10 files, 71 KiB; a monitoring query takes 93 ms


,monitoring action,median_ms,max_ms
0,write run metadata row (ingestion_runs),15.99,27.40
1,write validation report rows (validation_reports),8.49,10.31
2,storage scan of silver.tripdata (get_dir_size_...,0.29,2.86
3,storage scan of silver.weather (get_dir_size_b...,0.08,0.43


## 3. Incremental update vs full re-ingestion

What the incremental pipeline saves: the same Task 4 pipeline re-ingests *all* source files of the dataset
(original release + update) into an empty Silver table — what a platform without incremental processing would do.
Run once per dataset (it is long); `taxi_zone_lookup` is kept because the reference rule needs it.

In [8]:
RUN_FULL_REINGEST = True
full_rows = []
if RUN_FULL_REINGEST:
    for ds in DATASETS:
        prepare_run([])                                              # empty Silver, only taxi_zone_lookup
        files = sorted(__import__("glob").glob(CONFIGS[ds].absolute_raw_path))
        t0 = time.perf_counter()
        inserted = 0
        for path in files:
            m = run_validated_ingestion(ds, path, paths=RUN, reference=RUN, record=False)
            assert m["status"] == "SUCCESS", m["error_message"]
            inserted += m["records_valid_silver"]
        full_rows.append({"dataset": ds, "files": len(files), "full_reingest_s": time.perf_counter() - t0,
                          "rows_in_silver": inserted})
        print(f"{ds}: {len(files)} files, {full_rows[-1]['full_reingest_s']:.1f}s")

    full = pd.DataFrame(full_rows).set_index("dataset")
    full["incremental_s"] = ingest_median["task4+monitoring"]
    full["speed-up"] = (full["full_reingest_s"] / full["incremental_s"]).round(1)
    full = full.round(2)
    full.to_csv(os.path.join(RESULTS_DIR, "incremental_vs_full_ingestion.csv"))
    display(full)

tripdata: 4 files, 102.4s
weather: 2 files, 3.0s


air_quality: 2 files, 59.5s


,files,full_reingest_s,rows_in_silver,incremental_s,speed-up
dataset,,,,,
tripdata,4,102.43,9848033,11.78,8.7
weather,2,2.99,8952,1.81,1.7
air_quality,2,59.48,8140025,3.45,17.2


## 4. Analytical refresh time

After the three updates are ingested (un-timed), the analytical layer is brought up to date in two ways,
from the same Silver input:

* **incremental** — `build_integrated_taxi_trips_incremental` (only trips not yet in Gold, appended) followed by the
  Task 2 `refresh_products_incremental` (recompute only the touched groups, `MERGE` into the products);
* **full** — `build_integrated_taxi_trips` (whole Gold table rebuilt) followed by the Week 2 product build.

The integration module and the Task 2 refresh read their paths from module / notebook globals, which are pointed at
the `RUN` copy.

In [9]:
def point_analytics_to(paths, gold_dir, products_dir):
    integration.TRIPDATA_SILVER = paths.table("silver", "tripdata")
    integration.WEATHER_SILVER = paths.table("silver", "weather")
    integration.AIR_QUALITY_SILVER = paths.table("silver", "air_quality")
    integration.ZONE_LOOKUP_SILVER = paths.table("silver", "taxi_zone_lookup")
    integration.GOLD_PATH = gold_dir
    T2["GOLD_PATH"] = gold_dir
    T2["PRODUCTS_DIR"] = products_dir
    T2["record_pipeline_run"] = lambda m: write_deltalake(
        os.path.join(paths.root, "metadata", "pipeline_runs"), pd.DataFrame([asdict(m)]), mode="append", schema_mode="merge")


def refresh_cycle():
    prepare_run(with_gold=True)
    for ds in DATASETS:                                             # bring Silver up to date (not timed)
        ingest_task4(ds, UPDATE_FILES[ds], record=False)

    result = {}
    # incremental
    point_analytics_to(RUN, gold_path(RUN), products_path(RUN))
    gold_before = get_dir_size_bytes(gold_path(RUN))
    t0 = time.perf_counter()
    new_gold, n_new = integration.build_integrated_taxi_trips_incremental(spark)
    write_delta(new_gold, gold_path(RUN), mode="append", partition_by=["pickup_year", "pickup_month"])
    result["gold_incremental_s"] = time.perf_counter() - t0
    result["gold_incremental_rows"] = n_new
    result["gold_incremental_bytes"] = get_dir_size_bytes(gold_path(RUN)) - gold_before
    result["products_incremental_s"], _ = timed(T2["refresh_products_incremental"], spark, new_gold)

    # full recompute from the same Silver
    full_gold, full_products = os.path.join(RUN.root, "gold_full", GOLD), os.path.join(RUN.root, "gold_full", PRODUCTS)
    point_analytics_to(RUN, full_gold, full_products)
    t0 = time.perf_counter()
    result["gold_full_rows"] = write_delta(integration.build_integrated_taxi_trips(spark), full_gold,
                                           mode="overwrite", partition_by=["pickup_year", "pickup_month"])
    result["gold_full_s"] = time.perf_counter() - t0
    result["gold_full_bytes"] = get_dir_size_bytes(full_gold)
    result["products_full_s"], _ = timed(build_products_full, read_delta_spark(full_gold), full_products)

    # both paths must give the same products
    diffs = {}
    for name in week2_products(read_delta_spark(full_gold)):
        inc = read_delta_spark(os.path.join(products_path(RUN), name)).drop("_created_at", "_updated_at")
        ful = read_delta_spark(os.path.join(full_products, name)).drop("_created_at", "_updated_at")
        diffs[name] = inc.exceptAll(ful).count() + ful.exceptAll(inc).count()
    result["product_rows_that_differ"] = sum(diffs.values())
    return result


refresh_rows = []
for rep in range(REPEATS):
    refresh_rows.append({"repeat": rep, **refresh_cycle()})
    print(f"cycle {rep}: done")
refresh = pd.DataFrame(refresh_rows)
refresh.to_csv(os.path.join(RESULTS_DIR, "analytical_refresh.csv"), index=False)

refresh_summary = pd.DataFrame({
    "incremental_s": [refresh["gold_incremental_s"].median(), refresh["products_incremental_s"].median()],
    "full_s": [refresh["gold_full_s"].median(), refresh["products_full_s"].median()],
}, index=["Gold (integrated_taxi_trips)", "data products (4 tables)"])
refresh_summary.loc["total"] = refresh_summary.sum()
refresh_summary["speed-up"] = refresh_summary["full_s"] / refresh_summary["incremental_s"]
display(refresh_summary.round(2))
display(refresh[["gold_incremental_rows", "gold_full_rows", "gold_incremental_bytes", "gold_full_bytes",
                 "product_rows_that_differ"]].head(1))

/home/bogdan/repos/ID2221/LAB/.venv/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
                                                                                

daily_mobility_summary: merged 108 affected group(s)
taxi_zone_statistics: merged 245 affected group(s)
weather_impact_summary: merged 8 affected group(s)
air_quality_impact_summary: merged 2 affected group(s)


/home/bogdan/repos/ID2221/LAB/.venv/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
                                                                                

cycle 0: done


/home/bogdan/repos/ID2221/LAB/.venv/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
                                                                                

daily_mobility_summary: merged 108 affected group(s)


... (5 outputs left out)

/home/bogdan/repos/ID2221/LAB/.venv/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
                                                                                

cycle 2: done


,incremental_s,full_s,speed-up
Gold (integrated_taxi_trips),15.52,17.45,1.12
data products (4 tables),24.79,1.54,0.06
total,40.31,18.99,0.47


,gold_incremental_rows,gold_full_rows,gold_incremental_bytes,gold_full_bytes,product_rows_that_differ
0,293655,9848431,28174282,1029874469,2


## 5. Storage overhead

Every platform Delta table (read-only), split into

* **active** — files referenced by the current version (the data itself);
* **history** — files no longer referenced but kept for time travel (reclaimable with `VACUUM`);
* **log** — the `_delta_log` transaction log.

Week 3 adds the history kept by appends/merges/overwrites, the quarantine tables, the monitoring tables
(`metadata/*`) and the Bronze copies of the update files.

In [10]:
import pyarrow.compute as pc


def table_storage(path):
    dt = DeltaTable(path)
    adds = pa.table(dt.get_add_actions(flatten=True))
    total = get_dir_size_bytes(path)
    log = get_dir_size_bytes(os.path.join(path, "_delta_log"))
    active = int(pc.sum(adds["size_bytes"]).as_py() or 0)
    return {"versions": dt.version() + 1, "files": adds.num_rows,
            "rows": int(pc.sum(adds["num_records"]).as_py() or 0),
            "active_MB": active / MB, "history_MB": (total - active - log) / MB, "log_MB": log / MB,
            "total_MB": total / MB}


storage_rows = []
for layer in ["bronze", "silver", "quarantine", "gold", os.path.join("gold", PRODUCTS), "metadata"]:
    layer_dir = os.path.join(PLATFORM.root, layer)
    for name in sorted(os.listdir(layer_dir)) if os.path.isdir(layer_dir) else []:
        path = os.path.join(layer_dir, name)
        if not os.path.isdir(os.path.join(path, "_delta_log")):
            continue
        try:
            storage_rows.append({"layer": layer, "table": name, **table_storage(path)})
        except Exception as exc:  # noqa: BLE001 - a broken table is reported, not fatal
            storage_rows.append({"layer": layer, "table": name, "error": str(exc)[:80]})

storage = pd.DataFrame(storage_rows)
storage.to_csv(os.path.join(RESULTS_DIR, "storage_platform.csv"), index=False)
display(storage.round(3))

totals = storage.groupby("layer")[["active_MB", "history_MB", "log_MB", "total_MB"]].sum()
totals.loc["platform"] = totals.sum()
display(totals.round(2))

,layer,table,versions,files,rows,active_MB,history_MB,log_MB,total_MB
0,bronze,air_quality,9,3,8140551,13.351,0.052,0.037,13.440
1,bronze,taxi_zone_lookup,1,1,265,0.007,0.000,0.002,0.010
2,bronze,tripdata,11,5,10241944,202.050,20.390,0.028,222.468
3,bronze,weather,9,3,9120,0.083,0.036,0.031,0.150
4,silver,air_quality,6,4,8140025,62.344,0.058,0.039,62.441
5,silver,taxi_zone_lookup,1,1,265,0.008,0.000,0.002,0.010
6,silver,tripdata,11,18,9848448,916.617,72.443,0.071,989.131
7,silver,weather,9,3,8954,0.160,0.046,0.037,0.242
8,quarantine,air_quality,6,0,0,0.000,0.000,0.020,0.020
9,quarantine,taxi_zone_lookup,1,0,0,0.000,0.000,0.001,0.001


,active_MB,history_MB,log_MB,total_MB
layer,,,,
bronze,215.49,20.48,0.10,236.07
gold,982.30,24.60,0.05,1006.96
gold/data_products,0.04,0.08,0.04,0.16
metadata,0.33,0.05,0.16,0.54
quarantine,0.01,0.02,0.07,0.10
silver,979.13,72.55,0.15,1051.82
platform,2177.30,117.78,0.56,2295.65


In [11]:
p = totals.loc["platform"]
week3_tables = storage[storage["layer"].isin(["quarantine", "metadata"])]["total_MB"].sum()

baseline_sizes = {
    f"silver.{ds}": get_dir_size_bytes(BASELINE.table("silver", ds)) / MB for ds in DATASETS
}
baseline_sizes[f"gold.{GOLD}"] = get_dir_size_bytes(gold_path(BASELINE)) / MB
growth = pd.DataFrame({
    "pre_update_MB": pd.Series(baseline_sizes),
    "platform_active_MB": pd.Series({f"{r.layer}.{r.table}": r.active_MB for r in storage.itertuples()
                                     if f"{r.layer}.{r.table}" in baseline_sizes}),
})
growth["added_by_update_MB"] = growth["platform_active_MB"] - growth["pre_update_MB"]

overhead_storage = pd.DataFrame([
    {"item": "retained history (time travel, reclaimable by VACUUM)", "MB": p["history_MB"]},
    {"item": "Delta transaction logs", "MB": p["log_MB"]},
    {"item": "Week 3 tables: quarantine + monitoring metadata", "MB": week3_tables},
    {"item": "Gold: bytes written by one incremental refresh", "MB": refresh["gold_incremental_bytes"].median() / MB},
    {"item": "Gold: bytes written by one full rebuild", "MB": refresh["gold_full_bytes"].median() / MB},
])
overhead_storage["% of active data"] = 100 * overhead_storage["MB"] / p["active_MB"]
overhead_storage.to_csv(os.path.join(RESULTS_DIR, "storage_overhead.csv"), index=False)
display(growth.round(2))
display(overhead_storage.round(2))

,pre_update_MB,platform_active_MB,added_by_update_MB
gold.integrated_taxi_trips,955.28,982.30,27.02
silver.air_quality,62.26,62.34,0.08
silver.tripdata,892.01,916.62,24.61
silver.weather,0.14,0.16,0.02


,item,MB,% of active data
0,"retained history (time travel, reclaimable by ...",117.78,5.41
1,Delta transaction logs,0.56,0.03
2,Week 3 tables: quarantine + monitoring metadata,0.64,0.03
3,Gold: bytes written by one incremental refresh,26.87,1.23
4,Gold: bytes written by one full rebuild,982.17,45.11


## 6. Summary

In [12]:
summary = pd.DataFrame([
    {"measurement": "incremental update time (3 update files, Task 4 pipeline)",
     "value": f"{ingest_median['task4+monitoring'].sum():.1f} s",
     "context": f"full re-ingestion: {full['full_reingest_s'].sum():.1f} s" if RUN_FULL_REINGEST else ""},
    {"measurement": "analytical refresh time (Gold + 4 products, incremental)",
     "value": f"{refresh_summary.loc['total', 'incremental_s']:.1f} s",
     "context": f"full rebuild: {refresh_summary.loc['total', 'full_s']:.1f} s"},
    {"measurement": "storage overhead (history + logs + Week 3 tables)",
     "value": f"{p['history_MB'] + p['log_MB'] + week3_tables:.1f} MB",
     "context": f"{100 * (p['history_MB'] + p['log_MB'] + week3_tables) / p['active_MB']:.3f}% of {p['active_MB']:.0f} MB active data"},
    {"measurement": "validation overhead (Task 4, tripdata update)",
     "value": f"{overhead.loc['tripdata', 'task4_validation_s']:.1f} s",
     "context": f"{overhead.loc['tripdata', 'task4_validation_%']:.0f}% over no validation; Week 1: "
                f"{overhead.loc['tripdata', 'week1_validation_%']:.0f}%"},
    {"measurement": "monitoring overhead (per run)",
     "value": f"{micro['median_ms'][:2].sum():.0f} ms metadata writes",
     "context": f"+ {micro['median_ms'][2]:.0f} ms storage scan of silver.tripdata; end-to-end tripdata: "
                f"{overhead.loc['tripdata', 'task4_monitoring_s']:+.2f} s"},
])
summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
display(summary)
print("CSV files in", os.path.relpath(RESULTS_DIR, PROJECT_ROOT))

,measurement,value,context
0,"incremental update time (3 update files, Task ...",17.0 s,full re-ingestion: 164.9 s
1,"analytical refresh time (Gold + 4 products, in...",40.3 s,full rebuild: 19.0 s
2,storage overhead (history + logs + Week 3 tables),119.0 MB,5.465% of 2177 MB active data
3,"validation overhead (Task 4, tripdata update)",5.0 s,71% over no validation; Week 1: 837%
4,monitoring overhead (per run),24 ms metadata writes,+ 0 ms storage scan of silver.tripdata; end-to...


CSV files in reports/week3_evaluation


## 7. Discussion

All times are medians of 3 runs after a warm-up, on one machine (Spark `local[*]`, 6 GB driver); treat them as
orders of magnitude, not as exact benchmarks. Raw values: `reports/week3_evaluation/`.

### Measured results

| Measurement | Result | Comparison |
|---|---|---|
| Incremental update time (3 update files, Task 4 pipeline with monitoring) | **17.0 s** (tripdata 11.8 s, air quality 3.5 s, weather 1.8 s) | full re-ingestion of the same datasets: 164.9 s → **9.7× faster** (tripdata 8.7×, air quality 17.2×, weather 1.7×) |
| Analytical refresh time (Gold + 4 data products) | incremental **40.3 s** (Gold 15.5 s, products 24.8 s) | full rebuild **19.0 s** (Gold 17.4 s, products 1.5 s): Gold is 1.1× faster incrementally, the products are 16× slower, see below |
| Storage overhead of the updated platform | operational overhead **1.2 MB** (Delta logs 0.56 MB, quarantine + monitoring tables 0.64 MB); plus **117.8 MB** of retained history left by a rollback | 0.06 % resp. 5.4 % of 2,177 MB active data; the update itself added 24.6 MB to Silver tripdata and 27.0 MB to Gold |
| Validation overhead (Task 4 vs no validation) | tripdata **+5.1 s (+71 %)**, air quality +3.6 s, weather +1.2 s | Week 1 validation: tripdata +59.6 s (+837 %), air quality +112.5 s, weather +1.3 s |
| Monitoring overhead | **≈ 24 ms per run** (run-metadata row 16.0 ms + validation-report rows 8.5 ms; storage scan 0.3 ms) | ≈ 0.2 % of an 11.8 s tripdata run; the end-to-end comparison is dominated by a run-order effect (see below) |

**What the validation catches.** For the tripdata update (343,583 rows) the variants write different data:
without validation all 343,583 rows are inserted, including 49,911 trips that are already in Silver; Week 1 ignores
the duplicates and inserts 293,672 trips; Task 4 also ignores the duplicates, quarantines 17 trips (6 end before
they start, 4 last more than 24 hours, 7 have an impossible distance) and inserts 293,655, flagging 4,825 negative
fares as warnings. For air quality both validating variants ignore the 26 measurements already in Silver.

**Validation cost.** The percentages for the small files are inflated because `no_validation` takes under a second
for them; the absolute cost is ~1–5 s per file. Of the 6.26 s the 16 tripdata rules take together,
`duplicate_existing` (anti-join of the update against the 9.5 M keys already in Silver) costs 1.26 s; every other
rule costs ≤ 0.37 s. Validation cost is therefore driven by the size of the *existing* table, not by the size of the
update. The Week 1 implementation of the same duplicate check moves all existing keys through pandas and back into
Spark, which is why it costs 60 s for tripdata and 112 s for a 500-row air-quality file. The round trip is also
not exact: pandas timestamps come back into Spark in the driver's local time zone, so timestamp keys shift by the
UTC offset. Re-running the Week 1 ingestion of the weather update on this machine (UTC+2) inserted the last two
hours of the update a second time; the Task 4 anti-join, which stays in Spark, does not have this problem.

**Monitoring** is negligible per run. The end-to-end measurement even shows the `+monitoring` variants as *faster*
(tripdata Task 4 −0.4 s, air quality Task 4 −1.1 s, air quality Week 1 −3.5 s). That is an artefact of the run order, not
a real speed-up: within each repeat the variants always run as `no_validation → week1 → week1+monitoring → task4 →
task4+monitoring`, so every `+monitoring` run comes directly after its twin and re-reads the same update file and baseline
table from a warm OS page cache, with the identical Spark plans already compiled; the twin, in turn, often follows a heavy
run (`task4` runs right after the Week 1 pandas round trip over 8–9.5 M keys). The gap follows the weight of the preceding
run, not the amount of monitoring work, and vanishes where all runs are short (weather: ±0.1 s). The only work
`record=True` adds is two small Delta appends and a directory scan, which the micro-benchmark puts at ≈ 24 ms, far below
the run-to-run spread (e.g. `task4` on tripdata varied between 12.1 s and 13.2 s). The end-to-end numbers therefore only
show that monitoring overhead is below the measurement noise; the micro-benchmark is the figure to report. Randomising
the variant order per repeat would remove the bias.

The real cost of monitoring is structural: every run appends one small file to `ingestion_runs` (74 table versions,
62 files, 0.51 MB); in the micro-benchmark a query over only 10 such runs already took ~0.1 s, so the monitoring
tables need periodic compaction.

**Analytical refresh.** The incremental path appends 293,655 trips (26.9 MB) to Gold instead of rewriting 982 MB
(9.85 M rows), and produces the same products as a full rebuild (the 2–4 differing rows are last-decimal rounding
differences in groups the update does not touch, caused by floating-point summation order). For Gold it is only
slightly faster (15.5 s vs 17.4 s) because `build_integrated_taxi_trips_incremental` still reads the whole Silver trip
table and anti-joins it against all 9.5 M trip ids already in Gold, so its cost grows with the table, not the update.
The Task 2 product refresh is much slower than a full build (24.8 s vs 1.5 s): it caches the entire Gold table,
collects the touched keys, recomputes every touched group from the full table (the update's trips spread over two
weeks and many pickup zones) and runs four Delta `MERGE`s through pandas, while a full build of four aggregates reads only the needed columns.
The refresh times also varied a lot between the three cycles (Gold 15.2–22.8 s, products 18.8–30.0 s). The
incremental strategy currently pays off in I/O and storage — a full rebuild writes 36× more data, and on a Delta
table keeps the previous 982 MB version as history until `VACUUM` — but not in time; see the redesign below.

**Storage.** Normal Week 3 operation adds almost no overhead: Silver and Gold only receive appends, Delta logs are
0.56 MB and the quarantine and monitoring tables 0.64 MB. The 117.8 MB of history (72.4 MB Silver tripdata, 24.6 MB
Gold, 20.4 MB Bronze tripdata) was created by restoring the tables to their pre-update versions while the update
release was being regenerated: `RESTORE` only changes the transaction log, so the removed files stay on disk until
`VACUUM`. It illustrates the cost of time travel and rollback: every rolled-back release is kept in full until the
history is vacuumed. Bronze (215 MB) keeps a compressed copy of every delivered file, including re-deliveries (the
update files were ingested twice), which is the price of being able to replay any release.

### Which design decisions from Week 1 simplified maintenance?

* **Configuration-driven ingestion (`datasets.yaml`).** The update release was ingested without code changes; schema
  evolution was two YAML lines (`humidity`, `aqi`). The Task 4 validation rules extend the same file and the same
  loader (`config/datasets-expanded.yaml`).
* **Common data model with deterministic keys.** `trip_id` is a hash of the trip's content and every temporal dataset
  has an `event_timestamp` key, so re-delivered records are found by a key lookup (49,911 taxi duplicates, 26
  air-quality duplicates) instead of by comparing whole rows.
* **Bronze / Silver / quarantine layering with lineage columns.** `source_file` on every row made it possible to
  reconstruct the pre-update state for this evaluation with a filter, and makes every quarantined row traceable to
  its delivery.
* **Delta Lake with append + `schema_mode="merge"` and date partitioning.** Updates append new files to the latest
  partitions and never rewrite existing data; new columns are added without migrations, and a faulty release could
  be rolled back with `RESTORE`.
* **A run-metadata table from the start.** Task 3 monitoring and the Task 4 reports extended it with new columns
  instead of introducing a new mechanism.

### Which components required the largest modifications?

* **Validation** — from one fixed function (`apply_quality_checks` with numeric bounds) to a rule registry, schema
  policies and three actions (quarantine / ignore / warn), plus ~40 rule instances in the configuration. The Week 1 checks covered only primary keys and a few numeric
  bounds; they did not check cross-column logic (dropoff after pickup), code lists, references to the zone lookup,
  or schema changes other than missing columns (no unexpected columns, no type changes).
* **The analytical layer** — Gold and the products were full rebuilds; Week 3 added incremental Gold appends and
  touched-group `MERGE`s for the products (Task 2). As measured above, this part is not yet efficient.
* **Duplicate detection against existing data** — Week 1's pandas round trip dominated incremental ingestion
  (60–112 s) and shifts timestamp keys by the local UTC offset; the Spark anti-join in Task 4 does the same check in
  1.3 s and exactly.
* **Infrastructure fixes** found while operating the platform: Spark must read Delta tables through the transaction
  log (a directory read also returns files removed by overwrites and restores), `deltalake`-written timestamps must
  not be inferred as `timestamp_ntz`, and the Spark Python workers must use the project's interpreter
  (`PYSPARK_PYTHON`, Python ≤ 3.12).

### How well does the platform support future datasets?

Well at the ingestion and validation layers. A new source needs a YAML entry (format, column mapping, types, primary
key, partitioning, schema policy, rules) and usually a short transformation module that builds `event_timestamp`
(the existing ones are 4–28 lines). The generic rules (`not_null`, `unique`, `not_in_table`, `cast_errors`) and the
schema checks apply with column names only; `reference` connects it to a lookup table; monitoring and quarantine work
unchanged. Less so at the analytical layer: the joins in `integrated_taxi_trips.py` and the product definitions and
their touched-key logic are written per dataset, so integrating a new dataset into Gold and the products is still
code work. Duplicate detection also scales with the size of the existing Silver table, which becomes the bottleneck
as data accumulates unless it is limited to the partitions an update touches.

### If we redesigned the platform today, what would we change?

1. **Validation from day one.** Rule-based validation in the configuration from Week 1, so every dataset and every
   release is checked by the same engine and new rules never require pipeline changes.
2. **Scope incremental work by partition and by run.** Deduplicate against only the partitions (dates) an update
   touches; select new trips for Gold by ingestion run (`source_file` / `ingestion_timestamp`, or Delta change data
   feed) instead of anti-joining millions of ids; update additive products (counts, sums) from the new rows only
   instead of recomputing groups from a cached copy of Gold.
3. **One Delta engine, no pandas round trips.** Use delta-spark (or read through the transaction log everywhere),
   Delta `MERGE` for upserts and keep key comparisons in Spark, instead of mixing `deltalake` writes, Spark directory
   reads and pandas conversions.
4. **Product definitions in code, once.** Define each product in `src/` and derive both the full and the incremental
   refresh from that single definition; today the Week 2 and Task 2 notebooks each carry their own copy.
5. **Operational hygiene.** Scheduled `OPTIMIZE`/`VACUUM` (after a retention period that still allows rollback),
   compaction of the monitoring tables, a pinned Python version and `PYSPARK_PYTHON` in the repository.